In [1]:
%load_ext autoreload
%autoreload 2
from scipy.stats import spearmanr
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
import numpy as np
import os
import yaml
from theme import COLORS, colorway, apply_chart_theme, rgba
from utils import ROOT_DIR, to_int_id, to_str_id
from analysis_utils import (
    df_systems,
    SYSTEM_TABLE,
    INSTANCE_TABLE,
    NORMALITY_TABLE,
    ENERGY_TABLE,
    COVERAGE_TABLE,
    QUALITY_TABLE,
    TYPE_MAP,
    KEY_SMELLS,
    KEY_SMELLS_FULL,
    check_instance,
    plot_counts,
    fetch_rows,
    generate_and_replace_table,
    _spearman_from_rows
 )

pio.kaleido.scope.mathjax = None

ARTIFACTS_DIR = os.path.join(ROOT_DIR, "data", "artifacts")
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")
TABLES_DIR = os.path.join(ARTIFACTS_DIR, "tables")
DESIGN_PATH = os.path.join(TABLES_DIR, "design_tables.tex")
RESULTS_PATH = os.path.join(TABLES_DIR, "results_tables.tex")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

with open("../config/target.yaml", "r") as f:
    config = yaml.safe_load(f)
    config.update({k.lower(): v for k, v in config.items()})

rows = fetch_rows()

In [2]:
lower_cov = [
    row["instance_id"]
    for row in rows
    if row["branches_rf"] < row["branches_or"] or row["mutants_rf"] < row["mutants_or"]
]
lower_cov if lower_cov else "No instances with lower coverage"

'No instances with lower coverage'

In [3]:
high_outliers = [row["instance_id"] for row in rows if row["outliers_pct"] >= 10]
high_outliers if high_outliers else "No instances with high outliers"

'No instances with high outliers'

In [4]:
# fetch avg energy of all instances
sum([(row["energy_mean_or"] + row["energy_mean_rf"]) / 2 for row in rows]) / len(rows)

426.0427125340617

In [5]:
save = True
col_fig, dist_fig = check_instance(instance_id=to_str_id(5))
if save:
    col_fig.write_image(os.path.join(FIGURES_DIR, f"analysis_stable.pdf"))
    dist_fig.write_image(os.path.join(FIGURES_DIR, f"analysis_dist.pdf"))

Pulling from ../data/results\AA_0005\energy\latest


OR: normal (p=0.11)
RF: normal (p=0.18)
Significant: True (p=0.00)


In [6]:
save = True
if save:
    generate_and_replace_table(
        rows=df_systems.to_dict(orient="records"),
        columns=SYSTEM_TABLE,
        caption=r"Systems used in this study.",
        label="tab:systems",
        tex_path=DESIGN_PATH,
        group_by_key=None,
        sort_keys=None,
    )

No table containing \label{tab:systems} was found. Added to the end.


In [7]:
save = True
fig = plot_counts()
if save:
    fig.write_image(f"{FIGURES_DIR}/instances_by_type.pdf")

In [8]:
save = True
if save:
    generate_and_replace_table(
        columns=INSTANCE_TABLE,
        caption=r"List of all instances.",
        label="tab:instances",
        tex_path=DESIGN_PATH,
        group_by_key=None,
        sort_keys=None,
    )

No table containing \label{tab:instances} was found. Added to the end.


In [9]:
save = True
if save:
    generate_and_replace_table(
        columns=NORMALITY_TABLE,
        caption=r"Statistics and normality by test smell. Read \( OR \) and \( RF \) as original and refactored samples. Summary rows report the cumulative sum for sample sizes and absolute outlier counts, and the mean for outlier percentages. All rows satisfy normality.",
        label="tab:normality",
        tex_path=RESULTS_PATH,
        sort_keys=("type", "instance_id"),
        group_by_key="type",
        group_separator=r"\midrule",
        group_footer=True,
        include_separator_before_footer=False,
    )

No table containing \label{tab:normality} was found. Added to the end.


In [10]:
save = True
if save:
    generate_and_replace_table(
        columns=ENERGY_TABLE,
        caption=r"Energy metrics by test smell. Subscripts indicate original (OR) and refactored (RF) versions, and \( \mu \) and \( \sigma \) denote mean and standard deviation, respectively. Bold values indicate statistical significance (\( p < 0.05 \)) or large effects (\( d > 0.8 \)). Summary rows report cumulative averages for means, pooled standard deviations for variances, and medians for percentage changes.",
        label="tab:energy",
        tex_path=RESULTS_PATH,
        sort_keys=("type", "instance_id"),
        group_by_key="type",
        group_separator=r"\midrule",
        group_footer=True,
        include_separator_before_footer=False,
    )

No table containing \label{tab:energy} was found. Added to the end.


In [11]:
def quality_row_color(row: dict) -> str:
    has_difference = (row["branches_or"] != row["branches_rf"]) or (
        row["mutants_or"] != row["mutants_rf"]
    )
    return "black" if has_difference else "muted"


save = True
if save:
    generate_and_replace_table(
        columns=COVERAGE_TABLE,
        caption=r"Coverage and mutation metrics by test smell. Read \( OR \) and \( RF \) as original and refactored samples, and \( \Delta = RF - OR \). Rows with changes between original and refactored versions (\( \Delta \neq 0 \)) are marked in black, and unchanged rows are marked in grey.",
        label="tab:coverage",
        tex_path=RESULTS_PATH,
        sort_keys=("type", "instance_id"),
        group_by_key="type",
        group_separator=r"\midrule",
        group_footer=True,
        include_separator_before_footer=False,
        row_color=quality_row_color,
    )

No table containing \label{tab:coverage} was found. Added to the end.


In [12]:
save = True
if save:
    generate_and_replace_table(
        columns=QUALITY_TABLE,
        caption=r"Quality metrics by test smell. Read \( OR \) and \( RF \) as original and refactored samples, and \( \Delta = RF - OR \).",
        label="tab:quality",
        tex_path=RESULTS_PATH,
        sort_keys=("type", "instance_id"),
        group_by_key="type",
        group_separator=r"\midrule",
        group_footer=True,
        include_separator_before_footer=False,
    )

No table containing \label{tab:quality} was found. Added to the end.


In [13]:
key_smell_rows = [
    dict(
        type=smell,
        type_full=TYPE_MAP[smell],
        energy_pct=np.median([i["energy_pct"] for i in rows if i["type"] == smell]),
    )
    for smell in KEY_SMELLS
]
save = True
if save:
    generate_and_replace_table(
        rows=key_smell_rows,
        columns=[
            dict(
                header=r"\textbf{Test smell}",
                key="type_full",
                fmt="str",
                italic=True,
            ),
            dict(
                header=r"\textbf{Energy change (\%)}",
                key="energy_pct",
                fmt=".1f",
                threshold_min=5,
                abs_threshold=True,
                show_sign=True,
            ),
        ],
        caption=r"Key test smells with their median energy changes (in percent). \textit{Ignored test} and \textit{Lazy test (JUnit~5)} show the largest effects.",
        label="tab:key_smells",
        tex_path=RESULTS_PATH,
        sort_keys=None,
        group_by_key=None,
    )

No table containing \label{tab:key_smells} was found. Added to the end.


In [14]:
corr_rows = [
    (
        dict(
            type=smell,
            type_full=TYPE_MAP[smell],
            median_energy_pct=np.median(
                [i["energy_pct"] for i in rows if i["type"] == smell]
            ),
            median_duration_pct=np.median(
                [i["duration_pct"] for i in rows if i["type"] == smell]
            ),
            corr=_spearman_from_rows(
                [i for i in rows if i["type"] == smell], "duration"
            ),
        )
    )
    for smell in TYPE_MAP.keys()
    if any(i["type"] == smell for i in rows)
]

save = True
if save:
    generate_and_replace_table(
        rows=corr_rows,
        columns=[
            dict(
                header=r"\textbf{Test smell}",
                key="type_full",
                fmt=lambda x: (rf"\textbf{{{x}}}" if x in KEY_SMELLS_FULL else x),
                italic=True,
                align="l",
            ),
            dict(
                header=r"\textbf{Energy change (\%)}",
                key="median_energy_pct",
                fmt=".1f",
                threshold_min=5,
                abs_threshold=True,
                show_sign=True,
            ),
            dict(
                header=r"\textbf{Time change (\%)}",
                key="median_duration_pct",
                fmt=".1f",
                threshold_min=5,
                abs_threshold=True,
                show_sign=True,
            ),
            dict(
                header=r"\textbf{Correlation}",
                key="corr",
                fmt=".2f",
                threshold_min=0.7,
                abs_threshold=True,
            ),
        ],
        caption=r"Median changes and correlation between energy and execution time, for each test smell. Positive values indicate increases after refactoring, negative values indicate decreases, and bold values highlight notable percentage differences (\( > 5 \% \)) and strong statistical relationships (\( \rho > 0.70 \)). Most key smells, shown in bold, show strong correlations.",
        label="tab:corr_time",
        tex_path=RESULTS_PATH,
        sort_keys=None,
        group_by_key=None,
    )

No table containing \label{tab:corr_time} was found. Added to the end.


In [15]:
# rho(energy, time)
rho_all, p_all = spearmanr(
    [row["energy_mean_or"] for row in rows] + [row["energy_mean_rf"] for row in rows],
    [row["duration_mean_or"] for row in rows]
    + [row["duration_mean_rf"] for row in rows],
)

# rho(Δ% energy, Δ% time)
rho_delta, p_delta = spearmanr(
    [row["energy_pct"] for row in rows], [row["duration_pct"] for row in rows]
)

# print(f"N instances: {len(rows)}")
print(f"Spearman rho(energy, time) = {rho_all:.4f} (p={p_all:.4g})")
print(f"Spearman rho(Δ% energy, Δ% time) = {rho_delta:.4f} (p={p_delta:.4g})")

Spearman rho(energy, time) = 0.7249 (p=2.914e-14)
Spearman rho(Δ% energy, Δ% time) = 0.7649 (p=9.103e-09)


In [16]:
# quadrant scatter: %Δ time vs %Δ energy
df_quadrant = pd.DataFrame(
    {
        "instance_id": [row["instance_id"] for row in rows],
        "duration_pct": [row["duration_pct"] / 100 for row in rows],
        "energy_pct": [row["energy_pct"] / 100 for row in rows],
        "type": [row.get("type_full") for row in rows],
    }
)
df_quadrant = df_quadrant.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["duration_pct", "energy_pct"]
)

# group types
keep_types = {"Ignored test", "Lazy test (JUnit~5)"}
df_quadrant["type_group"] = df_quadrant["type"].apply(
    lambda t: t if t in keep_types else "Other smells"
)

# color palette by type group
type_order = ["Ignored test", "Lazy test (JUnit~5)", "Other smells"]
type_colors = {
    "Ignored test": colorway[0],
    "Lazy test (JUnit~5)": colorway[1],
    "Other smells": COLORS.MUTED,
}

fig = go.Figure()
for t in type_order:
    df_t = df_quadrant[df_quadrant["type_group"] == t]
    if df_t.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=df_t["duration_pct"],
            y=df_t["energy_pct"],
            mode="markers",
            text=[str(i) for i in df_t["instance_id"]],
            textposition="top center",
            marker=dict(size=9, color=type_colors.get(t, COLORS.PRIMARY), opacity=0.85),
            name=str(t).replace("~", " "),
            hoverinfo="none",
        )
    )

# quadrant lines at 0% (below markers)
fig.add_shape(
    type="line",
    x0=0,
    x1=0,
    y0=-1,
    y1=1,
    xref="x",
    yref="y",
    line=dict(color=COLORS.MUTED, width=1, dash="dot"),
    layer="below",
)
fig.add_shape(
    type="line",
    x0=-1,
    x1=1,
    y0=0,
    y1=0,
    xref="x",
    yref="y",
    line=dict(color=COLORS.MUTED, width=1, dash="dot"),
    layer="below",
)

# axis ranges centered around 0 with symmetric bounds
x_max = np.nanmax(np.abs(df_quadrant["duration_pct"])) if not df_quadrant.empty else 1.0
y_max = np.nanmax(np.abs(df_quadrant["energy_pct"])) if not df_quadrant.empty else 1.0
x_lim = max(0.1, float(x_max) * 1.1)
y_lim = max(0.1, float(y_max) * 1.1)

p_val = f" = {p_delta:.3g}" if p_delta > 0.001 else " < 0.001"

# correlation for only the two highlighted groups
highlight_mask = df_quadrant["type_group"].isin(keep_types)
df_highlight = df_quadrant[highlight_mask]
if len(df_highlight) >= 2:
    rho_highlight, p_highlight = spearmanr(
        df_highlight["duration_pct"], df_highlight["energy_pct"]
    )
else:
    rho_highlight, p_highlight = np.nan, np.nan

if np.isnan(rho_highlight) or np.isnan(p_highlight):
    highlight_text = "Highlighted groups: ρ = n/a"
else:
    p_highlight_val = f" = {p_highlight:.3g}" if p_highlight > 0.001 else " < 0.001"
    highlight_text = f"Highlighted groups: ρ = {rho_highlight:.3f} (p{p_highlight_val})"

fig.add_annotation(
    x=0.02,
    y=0.98,
    xref="paper",
    yref="paper",
    text=f"All groups: ρ = {rho_delta:.3f} (p{p_val})",
    showarrow=False,
    align="left",
    font=dict(color=COLORS.BLACK, size=12),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor=COLORS.MUTED,
    borderwidth=1,
    borderpad=4,
)
fig.add_annotation(
    x=0.02,
    y=0.91,
    xref="paper",
    yref="paper",
    text=highlight_text,
    showarrow=False,
    align="left",
    font=dict(color=COLORS.BLACK, size=12),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor=COLORS.MUTED,
    borderwidth=1,
    borderpad=4,
)

fig.update_layout(
    # title="Quadrant scatter: %Δ time vs %Δ energy",
    xaxis=dict(
        title="Time (Δ%)",
        tickformat=".0%",
        range=[-x_lim, x_lim],
        zeroline=False,
        showline=True,
    ),
    yaxis=dict(
        title="Energy (Δ%)",
        tickformat=".0%",
        range=[-y_lim, y_lim],
        zeroline=False,
        showline=True,
    ),
    width=900,
    height=600,
    showlegend=True,
)

fig = apply_chart_theme(
    fig,
    overrides={
        "margin_t": 80,
        "margin_b": 80,
        "margin_l": 100,
        "margin_r": 40,
        "static_plot": True,
        "x_showline": True,
        "y_showline": True,
        "x_tick_pos": "outside",
        "y_tick_pos": "outside",
        "x_tick_len": 6,
        "y_tick_len": 6,
    },
)
fig.show()

save = True
if save:
    fig.write_image(f"{FIGURES_DIR}/energy_time_scatter.pdf")

In [17]:
def plot_boxplots_by_type(type: str, max_cols: int = 6, save=False):
    type_rows = [r for r in rows if r.get("type") == type]
    if not type_rows:
        print(f"No rows found for type: {type}")
        return None

    # stable ordering by numeric instance id
    type_rows = sorted(type_rows, key=lambda r: r["instance_id"])

    n_instances = len(type_rows)
    n_cols = min(max_cols, n_instances)
    n_rows = int(np.ceil(n_instances / n_cols))

    # empty subplot titles
    subplot_titles = [""] * len(type_rows)
    specs = [[{"secondary_y": True} for _ in range(n_cols)] for _ in range(n_rows)]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        shared_yaxes=False,
        horizontal_spacing=0.06,
        vertical_spacing=0,
        specs=specs,
    )

    # shared hidden duration scale (seconds) across all subplots
    duration_vals = []
    for r in type_rows:
        for key in ("duration_mean_or", "duration_mean_rf"):
            v = r.get(key)
            if v is not None and np.isfinite(v):
                duration_vals.append(float(v))

    # lock duration axis to [0, max]
    duration_max = max(duration_vals) if duration_vals else 1.0
    duration_range = [0.0, max(1.0, round(duration_max, 1) * 1.2)]

    for i, r in enumerate(type_rows):
        row_idx = (i // n_cols) + 1
        col_idx = (i % n_cols) + 1

        df_or = r.get("df_or")
        df_rf = r.get("df_rf")

        vals_or = []
        vals_rf = []

        if isinstance(df_or, pd.DataFrame) and "energy" in df_or.columns:
            vals_or = pd.to_numeric(df_or["energy"], errors="coerce").dropna().tolist()

        if isinstance(df_rf, pd.DataFrame) and "energy" in df_rf.columns:
            vals_rf = pd.to_numeric(df_rf["energy"], errors="coerce").dropna().tolist()

        fig.add_trace(
            go.Box(
                x=["OR"] * len(vals_or),
                y=vals_or,
                name="Original",
                legendgroup="OR",
                showlegend=(i == 0),
                marker_color=COLORS.PRIMARY,
                boxpoints="outliers",
                hoverinfo="none",
            ),
            row=row_idx,
            col=col_idx,
            secondary_y=False,
        )

        fig.add_trace(
            go.Box(
                x=["RF"] * len(vals_rf),
                y=vals_rf,
                name="Refactored",
                legendgroup="RF",
                showlegend=(i == 0),
                marker_color=COLORS.SECONDARY,
                boxpoints="outliers",
                hoverinfo="none",
            ),
            row=row_idx,
            col=col_idx,
            secondary_y=False,
        )

        # duration markers on hidden secondary y-axis
        dur_or = r.get("duration_mean_or")
        dur_rf = r.get("duration_mean_rf")

        dur_vals = []
        dur_text = []
        for v in (dur_or, dur_rf):
            if v is None or not np.isfinite(v):
                dur_vals.append(np.nan)
                dur_text.append("N/A")
            else:
                fv = float(v)
                dur_vals.append(fv)
                dur_text.append(f"{fv:.2f}s")

        fig.add_trace(
            go.Scatter(
                x=["OR", "RF"],
                y=dur_vals,
                mode="markers+text",
                name="Duration",
                legendgroup="DURATION",
                showlegend=(i == 0),
                text=dur_text,
                textposition="top center",
                textfont=dict(size=11, color=COLORS.BLACK),
                marker=dict(size=6, color=COLORS.BLACK),
                hoverinfo="skip",
            ),
            row=row_idx,
            col=col_idx,
            secondary_y=True,
        )

        # remove OR/RF tick labels; show instance id as subplot x-axis label
        fig.update_xaxes(
            title_text=f"Instance {r['instance_id']}",
            showticklabels=False,
            ticks="",
            row=row_idx,
            col=col_idx,
        )

        # left y-axis (energy)
        fig.update_yaxes(
            title_text=None,
            row=row_idx,
            col=col_idx,
            secondary_y=False,
        )

        # secondary y-axis label off
        fig.update_yaxes(
            title_text=None,
            row=row_idx,
            col=col_idx,
            secondary_y=True,
        )

    fig.update_layout(
        width=200 * n_cols,
        height=320 * n_rows + 100,
        title=f"{TYPE_MAP.get(type)} [{type}]" if not save else None,
        yaxis_title="Energy (J)",
    )

    fig = apply_chart_theme(
        fig,
        overrides={
            "static_plot": True,
            "margin_t": 120,
            "margin_b": 60,
            "margin_l": 80,
            "margin_r": 0,
            "legend_orientation": "v",
            "legend_yanchor": "bottom",
            "legend_xanchor": "right",
            "legend_x": 0.9,
            "legend_y": 1.1,
            "y_showline": True,
            "y_tick_pos": "outside",
            "y_tick_len": 6,
        },
    )

    # enforce hidden duration axes after theme application
    for r_i in range(1, n_rows + 1):
        for c_i in range(1, n_cols + 1):
            fig.update_yaxes(
                row=r_i,
                col=c_i,
                secondary_y=True,
                autorange=False,
                range=duration_range,
                tick0=0,
                dtick=1,
                showgrid=False,
                zeroline=False,
                showline=False,
                showticklabels=False,
                ticks="",
                visible=False,
            )

    if not save:
        fig.show()
    return fig


save = True
for type in KEY_SMELLS:
    fig = plot_boxplots_by_type(type, save=save)
    if save:
        fig.write_image(f"{FIGURES_DIR}/energy_box_{type.replace('/', '-')}.pdf")

In [18]:
def plot_stacked_energy_dram_by_type(type: str, save=False):
    type_rows = [r for r in rows if r.get("type") == type]
    if not type_rows:
        print(f"No rows found for type: {type}")
        return None

    type_rows = sorted(type_rows, key=lambda r: r["instance_id"])

    e_or = [r["energy_mean_or"] for r in type_rows]
    d_or = [r["dram_energy_mean_or"] for r in type_rows]
    e_rf = [r["energy_mean_rf"] for r in type_rows]
    d_rf = [r["dram_energy_mean_rf"] for r in type_rows]

    # build % change from OR energy to RF energy
    pct_labels = []
    for a, b in zip(e_or, e_rf):
        if a is None or a == 0:
            pct_labels.append("N/A")
        else:
            pct_labels.append(f"{((b - a) / a) * 100:+.1f}%")

    # numeric x positions to control spacing precisely:
    # - no gap between OR and RF in the same instance
    # - visible gap between different instances
    group_width = 0.80
    group_gap = 0.35
    bar_width = group_width / 2

    centers = np.arange(len(type_rows)) * (group_width + group_gap)
    x_or = centers - (bar_width / 2)
    x_rf = centers + (bar_width / 2)

    tickvals = centers
    ticktext = [f"Instance {str(r['instance_id'])}" for r in type_rows]

    fig = go.Figure()

    # OR
    fig.add_trace(
        go.Bar(
            x=x_or,
            y=e_or,
            width=bar_width,
            name="Original (PKG)",
            marker_color=COLORS.PRIMARY,
            showlegend=True,
            legendrank=1,
            hoverinfo="skip",
        )
    )
    fig.add_trace(
        go.Bar(
            x=x_or,
            y=d_or,
            width=bar_width,
            name="Original (DRAM)",
            marker_color=rgba(COLORS.PRIMARY, alpha=0.5),
            showlegend=True,
            legendrank=2,
            hoverinfo="skip",
        )
    )

    # RF
    fig.add_trace(
        go.Bar(
            x=x_rf,
            y=e_rf,
            width=bar_width,
            name="Refactored (PKG)",
            marker_color=COLORS.SECONDARY,
            showlegend=True,
            legendrank=3,
            hoverinfo="skip",
        )
    )
    fig.add_trace(
        go.Bar(
            x=x_rf,
            y=d_rf,
            width=bar_width,
            name="Refactored (DRAM)",
            marker_color=rgba(COLORS.SECONDARY, alpha=0.5),
            showlegend=True,
            legendrank=4,
            text=pct_labels,
            textposition="outside",
            hoverinfo="skip",
        )
    )

    # y-axis range 0 to 1.2x max val
    y_candidates = [v for v in (e_rf + d_rf) if v is not None and np.isfinite(v)] + [
        v for v in (e_or + d_or) if v is not None and np.isfinite(v)
    ]
    y_max = max(y_candidates) if y_candidates else 1.0
    y_range = [0, y_max * 1.2]

    fig.update_layout(
        barmode="stack",
        bargroupgap=0.0,  # no gap between OR/RF bars in the same instance group
        bargap=0.2,  # gap between instance groups
        title=f"{TYPE_MAP.get(type)} [{type}]" if not save else None,
        width=800,
        height=500,
        legend=dict(traceorder="normal"),
        xaxis=dict(
            tickmode="array",
            tickvals=tickvals,
            ticktext=ticktext,
            tickcolor=COLORS.MUTED,
            tickfont=dict(color=COLORS.BLACK),
        ),
        yaxis=dict(
            title="Energy (J)",
            range=y_range,
            tick0=0,
            dtick=100,
        ),
    )

    fig = apply_chart_theme(
        fig,
        overrides={
            "static_plot": True,
            "margin_t": 90,
            "margin_b": 60 if type != "LT/5" else 80,
            "legend_orientation": "v",
            "legend_yanchor": "bottom",
            "legend_y": 1.05,
            "legend_x": 1.0,
            "legend_xanchor": "right",
            "x_line_width": 1,
            "y_line_width": 1,
            "y_autorange": False,
        },
    )

    # fig.update_xaxes(layer="below traces")

    if not save:
        fig.show()
    return fig


save = True
for type in KEY_SMELLS:
    fig = plot_stacked_energy_dram_by_type(type, save=save)
    if save:
        fig.write_image(f"{FIGURES_DIR}/energy_bar_{type.replace('/', '-')}.pdf")